In [31]:
import pandas as pd
import numpy as np

In [32]:
links = pd.read_csv('../dataset/links.csv')
movies = pd.read_csv('../dataset/movies.csv')
ratings = pd.read_csv('../dataset/ratings.csv')
movies = movies.merge(links, on='movieId')
movies.head()

,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


In [33]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
movies.head()


,movieId,title,genres,imdbId,tmdbId
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,114709,862.0
1,2,Jumanji (1995),Adventure Children Fantasy,113497,8844.0
2,3,Grumpier Old Men (1995),Comedy Romance,113228,15602.0
3,4,Waiting to Exhale (1995),Comedy Drama Romance,114885,31357.0
4,5,Father of the Bride Part II (1995),Comedy,113041,11862.0


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [35]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

In [36]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.shape

(9742, 9742)

In [37]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()


In [38]:
def recommend(movie_title, cosine_sim=cosine_sim):
    
    idx = indices[movie_title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]

    return movies['title'].iloc[movie_indices]
recommend('Toy Story (1995)')


1706                                       Antz (1998)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: object

In [39]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)


In [40]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix.fillna(0))


In [41]:
pickle.dump(user_similarity, open('../user_similarity.pkl','wb'))
pickle.dump(user_movie_matrix, open('../user_movie_matrix.pkl','wb'))
